# BDA & SML Lab 3
## Topic - Feature Engineering and Distributed Machine Learning using PySpark MLlib

## Name - Manan Chahal
## Roll No - 23102C0044

# Initialization

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [ ]:
spark = SparkSession.builder.appName("Titanic_ML").getOrCreate()
spark

# Importing the dataset

In [ ]:
df = spark.read.csv("/content/Titanic-Dataset.csv", header=True, inferSchema=True)
df.show()
print(df.count())

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [ ]:
for col_name in df.columns:
    null_count = df.filter(df[col_name].isNull()).count()
    print(f"Null count for {col_name}: {null_count}")

Null count for PassengerId: 0
Null count for Survived: 0
Null count for Pclass: 0
Null count for Name: 0
Null count for Sex: 0
Null count for Age: 177
Null count for SibSp: 0
Null count for Parch: 0
Null count for Ticket: 0
Null count for Fare: 0
Null count for Cabin: 687
Null count for Embarked: 2


## Using mean to replace missing values for age

In [ ]:
from pyspark.ml.feature import Imputer

print(df.filter(df.Age.isNull()).count())

imputer = Imputer(
    inputCols=['Age'],
    outputCols=["Age"],
    strategy="mean"
)
df = imputer.fit(df).transform(df)

print(df.filter(df.Age.isNull()).count())



177
0


# Using Mode to replace missing values

In [ ]:
mode_embarked = (
    df.groupBy("Embarked")
      .count()
      .orderBy(desc("count"))
      .first()[0]
)

print("Mode of Embarked:", mode_embarked)

df = df.fillna({"Embarked": mode_embarked})

Mode of Embarked: S


## Encoding categorical values

In [ ]:
from pyspark.ml.feature import StringIndexer

si = StringIndexer(inputCols=['Sex', 'Embarked'],
                   outputCols=['Gender', 'Embarked_Indexed'])
df = si.fit(df).transform(df)
df.show()

+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+------+----------------+
|PassengerId|Survived|Pclass|                Name|   Sex|              Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Gender|Embarked_Indexed|
+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+------+----------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|             22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|   0.0|             0.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|             38.0|    1|    0|        PC 17599|71.2833|  C85|       C|   1.0|             1.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|             26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|   1.0|             0.0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|             35.0|    1|    0| 

# Creating a vector for each row

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Gender",
    "Embarked_Indexed"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data = assembler.transform(df)
data.select("features", "Survived").show(5, truncate=False)

+----------------------------------+--------+
|features                          |Survived|
+----------------------------------+--------+
|[3.0,22.0,1.0,0.0,7.25,0.0,0.0]   |0       |
|[1.0,38.0,1.0,0.0,71.2833,1.0,1.0]|1       |
|[3.0,26.0,0.0,0.0,7.925,1.0,0.0]  |1       |
|[1.0,35.0,1.0,0.0,53.1,1.0,0.0]   |1       |
|(7,[0,1,4],[3.0,35.0,8.05])       |0       |
+----------------------------------+--------+
only showing top 5 rows


# Scaling numerical columns and creating test train split

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)

scalerModel = scaler.fit(data)
data = scalerModel.transform(data)

In [ ]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Training Records:", train_data.count())
print("Testing Records:", test_data.count())

Training Records: 746
Testing Records: 145


# Traning the model

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="scaledFeatures",
    labelCol="Survived"
)

model = lr.fit(train_data)

In [ ]:
predictions = model.transform(test_data)

predictions.select(
    "Survived",
    "prediction",
    "probability"
).show()

+--------+----------+--------------------+
|Survived|prediction|         probability|
+--------+----------+--------------------+
|       1|       1.0|[0.36613276580852...|
|       0|       0.0|[0.71823097219064...|
|       1|       1.0|[0.42268866382856...|
|       0|       0.0|[0.96578081087334...|
|       1|       1.0|[0.33616737873769...|
|       1|       0.0|[0.54224788693181...|
|       0|       0.0|[0.90484596923561...|
|       0|       0.0|[0.70905470531663...|
|       0|       0.0|[0.90479079160968...|
|       0|       0.0|[0.88645476259810...|
|       1|       1.0|[0.28175233233617...|
|       0|       1.0|[0.37616559944443...|
|       0|       0.0|[0.87798942178673...|
|       1|       0.0|[0.55574654170351...|
|       0|       0.0|[0.70188399855925...|
|       0|       0.0|[0.94331182969496...|
|       0|       0.0|[0.90479079160968...|
|       0|       1.0|[0.38117928047482...|
|       0|       0.0|[0.88127671960974...|
|       0|       0.0|[0.95502938148759...|
+--------+-

# Getting Accuracy of the predictions

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="Survived",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.8137931034482758
